In [1]:
import numpy as np
import pandas as pd
import yaml
import math
from scipy.special import exp1
from IPython.display import display
import plotly.graph_objects as go

In [2]:
####################
### domain setup ###
####################

# number of wells
n_wells = 4

# number of individuals in the population in the decision variable space
num_reals = 150

# bounds of coordinate system
x_min, x_max = -50, 50
y_min, y_max = -50, 50

# wells cannot be within this many units of the centre (0,0)
no_go_radius = 10.0

# range of pumping rates [m3/day]
q_min, q_max = 6.5, 65

# aquifer properties
T_val = 100.0  # [m2/day]
S_val = 0.001  # [-]
t_eval_val = 365.0  # [day]

def get_s(Q, r, T, S, t):
    """Theis drawdown calculation"""
    if r <= 1e-3: return 0 
    u = (r**2 * S) / (4.0 * T * t)
    return (Q / (4.0 * math.pi * T)) * exp1(u)

##############################
### random well generation ###
##############################

valid_data = []
while len(valid_data) < num_reals:
    # generates random x, y, q for each well + random xc
    # structure: [w1_x, w1_y, w1_q, w2_x, w2_y, w2_q, ..., xc]
    row = []
    for i in range(n_wells):
        row.extend([np.random.uniform(x_min, x_max), 
                    np.random.uniform(y_min, y_max), 
                    np.random.uniform(q_min, q_max)])
    row.append(np.random.uniform(x_min, x_max)) # xc
    
    # check distance for all wells from centre
    coords = np.array(row[:-1]).reshape(n_wells, 3)[:, :2]
    distances = np.linalg.norm(coords, axis=1)
    
    if np.all(distances > no_go_radius):
        valid_data.append(row)

# select one random realization to analyze
idx = np.random.randint(0, num_reals)
selected_real = valid_data[idx]

# randomly select yc as either y_min or y_max to place the compliance point on the top or bottom boundary
yc = np.random.choice([y_min, y_max])
xc_val = selected_real[-1]

# organize well data
wells = []
for i in range(n_wells):
    wells.append({
        "X": selected_real[i*3],
        "Y": selected_real[i*3+1],
        "Q": selected_real[i*3+2]
    })

##########################################
### comparison of random wells vs. CoM ###
##########################################

# sums the drawdown contributions from all random wells at the compliance point (xc_val, yc)
s_total_random = 0
for w in wells:
    r = np.sqrt((w['X'] - xc_val)**2 + (w['Y'] - yc)**2)
    s_total_random += get_s(w['Q'], r, T_val, S_val, t_eval_val)

# calculates the pumping-rate weighted CoM of the random wells
total_q_random = sum(w['Q'] for w in wells)
com_x = sum(w['X'] * w['Q'] for w in wells) / total_q_random
com_y = sum(w['Y'] * w['Q'] for w in wells) / total_q_random

# calculates Q_equivalent for a single well at the centre of the domain (0,0)
# that produces the same drawdown at the compliance point (xc_val, yc) 
# as the random wells
r_centre_to_cp = np.sqrt(xc_val**2 + yc**2)
u_centre = (r_centre_to_cp**2 * S_val) / (4.0 * T_val * t_eval_val)
q_equiv_centre = (s_total_random * 4.0 * math.pi * T_val) / exp1(u_centre)

###############
### summary ###
###############

print(f"Analysis for Compliance Point at: ({xc_val:.2f}, {yc:.2f})")

# summarizes well data in a DataFrame for display
well_df_data = []
for i, w in enumerate(wells):
    well_df_data.append({"ID": f"Well {i+1}", "X": w['X'], "Y": w['Y'], "Q (m3/d)": w['Q']})

well_df_data.append({"ID": "CoM (Weighted)", "X": com_x, "Y": com_y, "Q (m3/d)": total_q_random})
well_df_data.append({"ID": "Centre Well", "X": 0.0, "Y": 0.0, "Q (m3/d)": q_equiv_centre})

df_wells = pd.DataFrame(well_df_data)
display(df_wells.style.format({"X": "{:.2f}", "Y": "{:.2f}", "Q (m3/d)": "{:.1f}"}).hide(axis='index'))

# verify that the CoM well produces the same drawdown at the compliance point as the random wells
results = [
    {"Scenario": "Original Random Wells", "Total Q": total_q_random, "Drawdown @ CP": s_total_random},
    {"Scenario": "Single Centre Well", "Total Q": q_equiv_centre, "Drawdown @ CP": get_s(q_equiv_centre, r_centre_to_cp, T_val, S_val, t_eval_val)}
]
display(pd.DataFrame(results).style.format({"Total Q (m3/d)": "{:.1f}", "Drawdown @ CP (m)": "{:.1f}"}))


Analysis for Compliance Point at: (20.67, 50.00)


ID,X,Y,Q (m3/d)
Well 1,-19.95,46.13,61.2
Well 2,49.98,4.41,55.8
Well 3,-19.05,-19.34,19.0
Well 4,14.50,47.25,41.8
CoM (Weighted),10.20,26.29,177.7
Centre Well,0.00,0.00,196.6


,Scenario,Total Q,Drawdown @ CP
0,Original Random Wells,177.745990,1.602154
1,Single Centre Well,196.613041,1.602154


In [ ]:
#####################
### visualisation ###
#####################

# calculates drawdown at the pit for the labels
s_pit_random = sum(get_s(w['Q'], np.sqrt(w['X']**2 + w['Y']**2), T_val, S_val, t_eval_val) for w in wells)
s_pit_centre_well = get_s(q_equiv_centre, 0, T_val, S_val, t_eval_val)

fig = go.Figure()

# creates a DataFrame from valid_data for easy plotting of ensembles
col_names = []
for i in range(1, n_wells + 1):
    col_names.extend([f"well_{i}_x", f"well_{i}_y", f"well_{i}_q"])
col_names.append("xc")
df_ensemble = pd.DataFrame(valid_data, columns=col_names)

# domain and no-go area
fig.add_shape(type="rect", x0=x_min, y0=y_min, x1=x_max, y1=y_max, line=dict(color="black"), opacity=0.1)
fig.add_shape(type="circle", x0=-no_go_radius, y0=-no_go_radius, x1=no_go_radius, y1=no_go_radius, line=dict(color="black", dash="dash"))

# compliance point
fig.add_trace(go.Scatter(
    x=[xc_val], y=[yc], mode='markers+text', 
    text=[f"Compliance Point<br>s: {s_total_random:.2f}m"],
    marker=dict(color='yellow', size=12, symbol='star', line=dict(width=1, color='black')), 
    name="Compliance Point", 
    textposition="top center" if yc == y_min else "bottom center"
))

# centre well (the drawdown matcher)
fig.add_trace(go.Scatter(
    x=[0], y=[0], mode='markers+text', 
    text=[f"Centre Well (0,0)<br>Q: {q_equiv_centre:.1f}"],
    marker=dict(color='black', size=10, symbol='hexagon'), 
    name="Centre Well", textposition="top center"
))

# colours for the wells and their ensembles
colours = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', '#FF6692', '#B6E880']

for i in range(n_wells):
    color = colours[i % len(colours)]
    # ensemble points
    fig.add_trace(go.Scatter(
        x=df_ensemble[f"well_{i+1}_x"], y=df_ensemble[f"well_{i+1}_y"],
        mode='markers', marker=dict(color=color, opacity=0.1), 
        name=f"Well {i+1} Ensemble", showlegend=False
    ))
    # selected well location
    w = wells[i]
    fig.add_trace(go.Scatter(
        x=[w['X']], y=[w['Y']], mode='markers+text', 
        text=[f"W{i+1}"], 
        marker=dict(color=color, size=10, line=dict(width=1, color='white')), 
        name=f"Well {i+1} Selected", textposition="top center"
    ))

# pumping CoM of the random wells
fig.add_trace(go.Scatter(
    x=[com_x], y=[com_y], mode='markers+text', 
    text=["CoM"], marker=dict(color='black', size=12, symbol='x'), 
    name="Pumping CoM", textposition="top center"
))


fig.update_layout(
    xaxis_title="X Coordinate", yaxis_title="Y Coordinate",
    template="plotly_white", width=800, height=800,
    title=(f"Drawdown Matching: Random Wells vs. Centre Well<br>"
           f"<sup>Required Q at Centre: {q_equiv_centre:.1f} m³/d to match s={s_total_random:.2f}m</sup>"),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.02)
)

fig.show()
